# Recomendación contextual de proveedores para administración

## Caso a resolver

El área administrativa recibe solicitudes de compra y debe seleccionar proveedores autorizados. La mejor opción depende del contexto: departamento, categoría, presupuesto, urgencia, ubicación, mes, costo, tiempo de entrega y cumplimiento histórico.

No es un caso de comercio electrónico. Es un problema de apoyo a decisiones internas y gestión de compras.

El notebook construye candidatos proveedor-solicitud, entrena un modelo supervisado y ordena los proveedores más convenientes para cada solicitud.

## 1. Técnica y método

### Técnica reconocida

Implementaremos un Context-Aware Recommender System mediante ranking contextual supervisado. La recomendación contextual no es un único algoritmo: es un enfoque que incorpora variables de la situación actual. Aquí la función de ranking será Gradient Boosting.

### Método

1. Generar solicitudes administrativas y proveedores autorizados.
2. Crear una fila por cada combinación solicitud-proveedor candidato.
3. Construir una variable objetivo: la opción seleccionada históricamente.
4. Entrenar GradientBoostingClassifier con variables de contexto y proveedor.
5. Calcular una probabilidad de elección para cada candidato.
6. Ordenar candidatos dentro de cada solicitud.
7. Recomendar el proveedor con mayor score y evaluar Top-1, Top-3 y MRR.

### Interpretación

El modelo no recomienda el producto más popular. Estima qué proveedor tiene mayor probabilidad de ser elegido dadas las condiciones específicas de la solicitud.

## 1.1 Fundamento de Gradient Boosting

Gradient Boosting construye muchos árboles pequeños de manera secuencial. Cada árbol intenta corregir los errores de los árboles anteriores. La combinación produce una función no lineal capaz de aprender interacciones, por ejemplo:

- una urgencia alta hace importante el tiempo de entrega;
- un presupuesto bajo hace importante el costo;
- una ubicación específica favorece proveedores regionales;
- una categoría especializada requiere mayor cumplimiento.

La salida del clasificador es una probabilidad o score. En recomendación, ese score se calcula para cada candidato y se utiliza para construir un ranking.

Parámetros principales:

- n_estimators: cantidad de árboles.
- learning_rate: cuánto contribuye cada árbol.
- max_depth: complejidad de cada árbol.
- random_state: reproducibilidad.

Importante: clasificación y recomendación no son exactamente lo mismo. La clasificación predice aceptación de cada candidato; la recomendación agrupa candidatos por solicitud y los ordena.

## 2. Configuración

Importamos generación de datos, transformación de variables categóricas, modelo, métricas y visualización. La semilla permite reproducir el ejercicio en Colab.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Entorno listo. Semilla:', RANDOM_STATE)

### Explicación detallada

OneHotEncoder transforma departamentos, categorías, ubicaciones y temporadas en variables binarias. ColumnTransformer aplica transformaciones a columnas seleccionadas. Pipeline conecta preparación y modelo para evitar errores de orden. GradientBoostingClassifier aprende la probabilidad de selección. Las métricas de clasificación y ranking miden el resultado.

## 3. Datos sintéticos administrativos

Generaremos 500 solicitudes y 6 proveedores autorizados. Cada solicitud se combina con todos los proveedores disponibles, creando candidatos.

Las variables de la solicitud serán departamento, categoría, presupuesto, urgencia, ubicación y mes. Las variables del proveedor serán costo estimado, días de entrega, cumplimiento histórico, cobertura regional y especialización.

In [ ]:
n_solicitudes = 500
proveedores = pd.DataFrame({
    'proveedor': ['A', 'B', 'C', 'D', 'E', 'F'],
    'costo_base_mxn': [750, 1100, 1800, 950, 1400, 2200],
    'dias_entrega_base': [8, 5, 3, 12, 7, 4],
    'cumplimiento_pct': [94, 97, 99, 88, 93, 96],
    'region': ['Norte', 'Centro', 'Centro', 'Sur', 'Norte', 'Centro'],
    'especialidad': ['general', 'tecnologia', 'mantenimiento', 'general', 'papeleria', 'tecnologia']
})
solicitudes = pd.DataFrame({
    'solicitud_id': [f'S-{i:04d}' for i in range(1, n_solicitudes+1)],
    'departamento': rng.choice(['Finanzas','Recursos Humanos','Operaciones','Mantenimiento'], n_solicitudes),
    'categoria': rng.choice(['papeleria','tecnologia','mantenimiento','servicios_generales'], n_solicitudes),
    'presupuesto_mxn': rng.uniform(1000, 15000, n_solicitudes).round(2),
    'urgencia': rng.choice(['baja','media','alta'], n_solicitudes, p=[.45,.40,.15]),
    'ubicacion': rng.choice(['Norte','Centro','Sur'], n_solicitudes),
    'mes': rng.integers(1, 13, n_solicitudes)
})
candidatos = solicitudes.merge(proveedores, how='cross')
candidatos['factor_urgencia'] = candidatos.urgencia.map({'baja':1.0,'media':1.4,'alta':2.0})
candidatos['costo_estimado_mxn'] = candidatos.costo_base_mxn * rng.uniform(.9, 1.15, len(candidatos))
candidatos['dias_entrega'] = (candidatos.dias_entrega_base * rng.uniform(.9, 1.2, len(candidatos))).round(1)
candidatos['ajuste_region'] = (candidatos.ubicacion == candidatos.region).astype(int)
candidatos['ajuste_categoria'] = (candidatos.categoria == candidatos.especialidad).astype(int)
candidatos['margen_presupuesto_pct'] = 100*(candidatos.presupuesto_mxn-candidatos.costo_estimado_mxn)/candidatos.presupuesto_mxn
utilidad = (2.5*candidatos.ajuste_categoria + 1.5*candidatos.ajuste_region
            + .08*candidatos.cumplimiento_pct - .35*candidatos.dias_entrega
            - .00015*candidatos.costo_estimado_mxn
            - .25*candidatos.factor_urgencia*candidatos.dias_entrega)
utilidad += rng.normal(0, .8, len(candidatos))
candidatos['utilidad'] = utilidad
ganadores = candidatos.groupby('solicitud_id')['utilidad'].transform('max')
candidatos['seleccionado_historico'] = (candidatos.utilidad == ganadores).astype(int)
print('Solicitudes:', len(solicitudes), '| Candidatos:', len(candidatos))
display(candidatos.head())

### Explicación detallada de la construcción

El catálogo de proveedores contiene atributos relativamente estables. Las solicitudes contienen el contexto que cambia en cada decisión. El producto cartesiano crea todas las alternativas posibles para cada solicitud.

La utilidad sintética representa una decisión histórica: favorece especialización, cobertura regional, cumplimiento y rapidez, penalizando costo y días de entrega. No se entrega utilidad al modelo; sólo se convierte en una selección histórica para simular ejemplos etiquetados.

## 4. Validación y exploración

Revisamos que no haya faltantes, que los costos y tiempos sean positivos y que cada solicitud tenga exactamente un proveedor seleccionado histórico.

In [ ]:
print('Faltantes:', candidatos.isna().sum().sum())
print('Costos no positivos:', int((candidatos.costo_estimado_mxn <= 0).sum()))
print('Tiempos no positivos:', int((candidatos.dias_entrega <= 0).sum()))
selecciones = candidatos.groupby('solicitud_id').seleccionado_historico.sum()
print('Solicitudes con una selección:', int((selecciones == 1).sum()), 'de', len(selecciones))
display(candidatos.groupby('proveedor')[['costo_estimado_mxn','dias_entrega','cumplimiento_pct']].mean().round(2))

### Interpretación de la validación

Cada solicitud debe tener una sola opción histórica elegida. Los controles físicos y administrativos garantizan que el modelo aprenda relaciones comerciales plausibles y no errores de datos. La tabla por proveedor ayuda a entender que un proveedor barato no necesariamente es el mejor si entrega tarde o tiene menor cumplimiento.

## 4.1 Visualizaciones del contexto

Visualizamos presupuesto, costo estimado, días de entrega y cumplimiento para entender la población antes del modelo.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
sns.histplot(candidatos, x='presupuesto_mxn', bins=30, ax=axes[0,0])
axes[0,0].set_title('Distribución del presupuesto')
sns.histplot(candidatos, x='costo_estimado_mxn', bins=30, ax=axes[0,1])
axes[0,1].set_title('Costo estimado por candidato')
sns.boxplot(data=candidatos, x='proveedor', y='dias_entrega', ax=axes[1,0])
axes[1,0].set_title('Tiempo de entrega por proveedor')
sns.boxplot(data=candidatos, x='proveedor', y='cumplimiento_pct', ax=axes[1,1])
axes[1,1].set_title('Cumplimiento por proveedor')
plt.tight_layout()
plt.show()

### Interpretación de las visualizaciones

Estas gráficas muestran por qué una recomendación contextual es preferible a una lista fija. El mejor proveedor puede cambiar con urgencia, presupuesto y ubicación. Los boxplots muestran consistencia y variabilidad; las distribuciones ayudan a detectar rangos poco plausibles antes del entrenamiento.

## 5. Preparación de variables

Separamos identificadores y la etiqueta. Las variables categóricas se codifican con OneHotEncoder y las numéricas pasan directamente al modelo.

In [ ]:
objetivo = 'seleccionado_historico'
no_usar = ['solicitud_id','proveedor','utilidad',objetivo]
caracteristicas = [c for c in candidatos.columns if c not in no_usar]
categoricas = ['departamento','categoria','urgencia','ubicacion','region','especialidad']
numericas = [c for c in caracteristicas if c not in categoricas]
preprocesador = ColumnTransformer([
    ('categoricas', OneHotEncoder(handle_unknown='ignore'), categoricas),
    ('numericas', 'passthrough', numericas)
])
X = candidatos[caracteristicas]
y = candidatos[objetivo]
print('Variables categóricas:', categoricas)
print('Variables numéricas:', numericas)
print('Filas para entrenamiento:', len(X))

### Explicación detallada

No usamos solicitud_id porque memorizaría casos, ni proveedor como identificador porque el modelo debe aprender atributos del proveedor. utilidad se elimina porque fue usada para generar la etiqueta y produciría fuga de información.

handle_unknown='ignore' permite procesar una categoría nueva sin romper el pipeline. Mantener preprocesador y modelo juntos garantiza que una recomendación nueva reciba exactamente la misma transformación.

## 6. Entrenamiento del modelo contextual

Usamos GradientBoostingClassifier. El modelo predice si un candidato sería seleccionado en el contexto de una solicitud.

In [ ]:
modelo = Pipeline([
    ('preprocesamiento', preprocesador),
    ('gradient_boosting', GradientBoostingClassifier(
        n_estimators=150, learning_rate=.05, max_depth=3, random_state=RANDOM_STATE
    ))
])
modelo.fit(X, y)
candidatos['score_recomendacion'] = modelo.predict_proba(X)[:, 1]
candidatos['prediccion_aceptacion'] = (candidatos['score_recomendacion'] >= .5).astype(int)
print('Modelo entrenado. Score agregado a cada candidato.')
display(candidatos[['solicitud_id','proveedor','score_recomendacion','seleccionado_historico']].head(10).round(3))

### Explicación detallada

El pipeline primero codifica categorías y luego ajusta los árboles. predict_proba produce un score comparable entre candidatos. Ese score no es necesariamente una probabilidad perfectamente calibrada; en este ejercicio se usa para ordenar alternativas.

La recomendación se obtiene después, al agrupar por solicitud. No basta con clasificar filas de manera independiente: debemos comparar proveedores que compiten por la misma solicitud.

## 7. Construcción del ranking y evaluación Top-K

Medimos si el proveedor seleccionado históricamente aparece como primera recomendación o dentro de las tres primeras.

In [ ]:
ranking = candidatos.sort_values(['solicitud_id','score_recomendacion'], ascending=[True, False]).copy()
ranking['posicion'] = ranking.groupby('solicitud_id').cumcount() + 1
top1 = ranking[ranking.posicion == 1]
top3 = ranking[ranking.posicion <= 3]
hit_rate_1 = top1.seleccionado_historico.mean()
hit_rate_3 = top3.groupby('solicitud_id').seleccionado_historico.max().mean()
rr = ranking[ranking.seleccionado_historico == 1].groupby('solicitud_id').posicion.first()
mrr = (1/rr).mean()
print(f'Hit Rate@1: {hit_rate_1:.1%}')
print(f'Hit Rate@3: {hit_rate_3:.1%}')
print(f'MRR: {mrr:.3f}')
display(ranking[['solicitud_id','posicion','proveedor','score_recomendacion','seleccionado_historico']].head(12).round(3))

### Interpretación ejecutiva de la recomendación

El modelo no responde simplemente cuál proveedor es el más popular. Para cada solicitud calcula un score considerando el contexto específico y después compara únicamente los proveedores que compiten por esa misma solicitud.

Por ejemplo, el proveedor recomendado para una solicitud urgente de tecnología puede no ser el mismo que para una compra rutinaria de papelería. La recomendación refleja esa diferencia porque incorpora urgencia, categoría, presupuesto, especialidad, tiempo de entrega, ubicación y cumplimiento.

La salida debe leerse como una lista ordenada de alternativas, no como una orden automática de compra.

### Interpretación detallada de Hit Rate@1, Hit Rate@3 y MRR

- **Hit Rate@1:** indica cuántas solicitudes reciben como primera opción el proveedor que aparece como seleccionado en el histórico sintético. Un valor alto significa que el sistema suele colocar una alternativa adecuada en la posición principal.
- **Hit Rate@3:** indica cuántas solicitudes tienen la opción histórica dentro de las tres primeras. Es útil cuando el comprador necesita comparar alternativas y no desea una única respuesta rígida.
- **MRR:** da más valor a los aciertos de las primeras posiciones. Un MRR alto significa que, incluso cuando el modelo no coloca la opción correcta en primer lugar, suele mantenerla cerca del inicio del ranking.

En un proceso administrativo, Hit Rate@3 puede ser más útil que Hit Rate@1 cuando existen restricciones de contrato, disponibilidad o autorización que no están en los datos. El sistema presenta tres opciones y el comprador toma la decisión final.

### Interpretación detallada del ranking

Hit Rate@1 indica en qué proporción de solicitudes el primer proveedor recomendado coincide con la elección histórica. Hit Rate@3 indica si la opción aparece entre las tres primeras. MRR asigna más valor a aciertos en posiciones superiores: un acierto en posición 1 vale 1, en posición 2 vale 0.5 y así sucesivamente.

Estas métricas son más adecuadas que accuracy para recomendación porque la decisión final consiste en ordenar alternativas. En una operación real, la etiqueta histórica puede reflejar sesgos o decisiones no óptimas; por eso también se deben medir costo, tiempo de entrega y satisfacción posterior.

## 8. Interpretación de recomendaciones

Mostramos recomendaciones para una solicitud y comparamos score con atributos de negocio.

In [ ]:
solicitud_ejemplo = ranking.solicitud_id.iloc[0]
display(ranking[ranking.solicitud_id == solicitud_ejemplo][
    ['solicitud_id','departamento','categoria','presupuesto_mxn','urgencia','ubicacion',
     'proveedor','costo_estimado_mxn','dias_entrega','cumplimiento_pct',
     'score_recomendacion','posicion']
].round(2))

### Cómo convertir el ranking en una recomendación accionable

La recomendación debe incluir cuatro elementos:

1. Proveedor sugerido y posición en el ranking.
2. Score relativo frente a las otras alternativas.
3. Razones observables: costo, entrega, cumplimiento, especialidad y región.
4. Advertencias: presupuesto insuficiente, proveedor no autorizado o información faltante.

Una diferencia pequeña entre el primer y segundo score indica que la recomendación es débil y conviene presentar ambas opciones. Una diferencia amplia indica mayor confianza relativa, aunque no elimina la necesidad de validar políticas internas.

### Interpretación de la recomendación mostrada

La tabla de la solicitud de ejemplo debe leerse fila por fila: la posición 1 es la alternativa recomendada, y las siguientes posiciones son opciones de respaldo. El proveedor en primera posición no necesariamente es el más barato; puede obtener un score superior por combinar mejor rapidez, cumplimiento, especialización y adecuación regional.

La decisión final debe validar que el proveedor esté autorizado, que el importe esté dentro del presupuesto y que exista capacidad de entrega. Si una restricción dura no está representada en las variables, debe aplicarse como filtro antes o después del ranking.

### Interpretación del resultado

La primera fila es la recomendación principal porque tiene el mayor score dentro de esa solicitud. El analista debe comprobar que sea compatible con presupuesto, autorización y cobertura. Una alternativa en segunda posición puede ser preferible si existe una restricción no incluida, como contrato vigente o capacidad disponible.

La recomendación debe presentarse con sus razones observables: costo, entrega, cumplimiento, especialidad y región. Esto aumenta la confianza y permite detectar cuando el modelo aprendió una relación incorrecta.

## 9. Importancia de variables

Gradient Boosting permite inspeccionar qué variables contribuyeron más al modelo después del preprocesamiento. Esta interpretación orienta, pero no equivale a causalidad.

In [ ]:
modelo_final = modelo.named_steps['gradient_boosting']
nombres = modelo.named_steps['preprocesamiento'].get_feature_names_out()
importancias = pd.Series(modelo_final.feature_importances_, index=nombres).sort_values(ascending=False).head(15)
display(importancias.to_frame('importancia'))
importancias.sort_values().plot(kind='barh', figsize=(9,5), title='Variables con mayor importancia')
plt.xlabel('Importancia relativa'); plt.show()

### Qué significan las variables importantes para la recomendación

Si días de entrega aparece entre las variables importantes, el modelo está utilizando la rapidez para ordenar proveedores. Si destaca ajuste de categoría, está reconociendo la especialización. Si aparece cumplimiento, está favoreciendo proveedores con mejor historial.

La importancia no significa causalidad ni garantiza que una variable sea suficiente. Sirve para explicar qué señales utiliza el modelo y para detectar resultados inesperados. En una revisión de gobierno del modelo, estas señales deben ser aprobadas por compras y administración.

### Riesgos y controles de la recomendación

Una recomendación puede ser técnicamente correcta y administrativamente inválida si ignora contratos, conflictos de interés, límites de autorización, disponibilidad o proveedores bloqueados. Por eso el sistema debe aplicar reglas de negocio y registrar quién aceptó o rechazó la sugerencia.

También debe evitar el sesgo de historial: si siempre se contrató al mismo proveedor, el modelo puede recomendarlo aunque una alternativa nueva tenga mejor desempeño. Conviene incluir exploración controlada y medir costo, tiempo, cumplimiento y satisfacción posteriores.

### Interpretación de importancia

Si aparecen días de entrega, cumplimiento, especialidad o costo entre las variables principales, el modelo está usando señales razonables para seleccionar proveedores. La importancia no dice que una variable cause la selección ni que sea suficiente por sí sola. Debe complementarse con pruebas de sensibilidad y validación con usuarios.

## 10. Conclusiones

- La recomendación contextual considera la situación completa de la solicitud, no sólo popularidad histórica.
- Gradient Boosting aprende relaciones no lineales entre contexto y atributos del proveedor.
- El ranking debe evaluarse con Hit Rate@1, Hit Rate@3 y MRR, no únicamente con accuracy.
- Las recomendaciones deben mostrar score y atributos de negocio para facilitar revisión.
- Las etiquetas históricas pueden contener sesgos; se requiere validación con decisiones reales y resultados posteriores.
- En producción se deben incorporar restricciones duras: proveedores autorizados, contratos, presupuesto, conflicto de interés y capacidad.

### Siguiente paso recomendado

Crear un conjunto de evaluación temporal y medir si la recomendación reduce costo, tiempo de compra y excepciones administrativas.

## 11. Resumen reproducible

In [ ]:
print({'solicitudes': len(solicitudes), 'candidatos': len(candidatos),
       'hit_rate_at_1': round(hit_rate_1, 3),
       'hit_rate_at_3': round(hit_rate_3, 3),
       'mrr': round(mrr, 3)})

### Interpretación ejecutiva de la recomendación

El modelo no responde simplemente cuál proveedor es el más popular. Para cada solicitud calcula un score considerando el contexto específico y después compara únicamente los proveedores que compiten por esa misma solicitud.

Por ejemplo, el proveedor recomendado para una solicitud urgente de tecnología puede no ser el mismo que para una compra rutinaria de papelería. La recomendación refleja esa diferencia porque incorpora urgencia, categoría, presupuesto, especialidad, tiempo de entrega, ubicación y cumplimiento.

La salida debe leerse como una lista ordenada de alternativas, no como una orden automática de compra.

### Interpretación detallada de Hit Rate@1, Hit Rate@3 y MRR

- **Hit Rate@1:** indica cuántas solicitudes reciben como primera opción el proveedor que aparece como seleccionado en el histórico sintético. Un valor alto significa que el sistema suele colocar una alternativa adecuada en la posición principal.
- **Hit Rate@3:** indica cuántas solicitudes tienen la opción histórica dentro de las tres primeras. Es útil cuando el comprador necesita comparar alternativas y no desea una única respuesta rígida.
- **MRR:** da más valor a los aciertos de las primeras posiciones. Un MRR alto significa que, incluso cuando el modelo no coloca la opción correcta en primer lugar, suele mantenerla cerca del inicio del ranking.

En un proceso administrativo, Hit Rate@3 puede ser más útil que Hit Rate@1 cuando existen restricciones de contrato, disponibilidad o autorización que no están en los datos. El sistema presenta tres opciones y el comprador toma la decisión final.